# `baselines_sota.ipynb` — 4 SOTA baselines on Llama-2-7B

Re-implementations of four published hallucination-detection methods on the SAME backbone (`Llama-2-7B`) and SAME train/test split as `all_variants.ipynb`, so the comparison is apples-to-apples. The point of this notebook is NOT to set new SOTA — it is to measure the gap between methods at a single fixed backbone scale where the model is small enough to fit Colab Free.

## Baselines (all 4 ported from official repos)

| Baseline | Source | What it does on Llama-2-7B |
|---|---|---|
| **SAPLMA** | [Azaria & Mitchell 2023, arXiv 2304.13734](https://arxiv.org/abs/2304.13734) | MLP probe `768→256→128→64→1` on layer-7 last-token hidden state. BCE, 5 epochs. |
| **HaloScope** | [Du et al. NeurIPS 2024](https://arxiv.org/abs/2409.17504) · [repo](https://github.com/deeplearning-wisc/haloscope) | Spectral score on last-token last-layer embeddings → top-k SVD direction → pseudo-labels by percentile threshold → non-linear probe `768→1024→1`. Layer × k × sign × threshold swept on validation. |
| **EigenScore** | [Chen et al. INSIDE, ICLR 2024](https://arxiv.org/abs/2402.03744) · [repo](https://github.com/D2I-ai/eigenscore) | Per query, sample K responses, take middle-layer (= EIGENSCORE_LAYER for Llama-2-7B) last-token hidden state of each. Compute K×K covariance + 1e-3 ridge. Score = `mean(log10(σ))`. Higher = more semantic diversity = more likely hallucinated. UNSUPERVISED. K=10 for Wikipedia held-out, K=3 with 100-sample cap for multi-task (cost). |
| **HalluShift** | [Dasgupta IJCNN 2025](https://arxiv.org/abs/2504.09482) · [repo](https://github.com/sharanya-dasgupta001/hallushift) | 31-d feature vector = 5 Wasserstein + 5 cosine on hidden states (offset r=2) + 5 Wasserstein + 5 cosine on attentions + 11 token-probability features (min P_max, max spread, normalized entropies, low-prob counts, mean gradients, percentiles). 4 parallel `LayerNorm→Dropout→Linear` heads → concat → 2-layer head → sigmoid. Loss = BCEWithLogits + 0.4·(1 - acc). AdamW lr=1e-4. |

## Methodological notes

* **Data reuse.** All baselines train on the same 80% split of `llama2_7b_dataset_full.json` (1000 records, 500 hallucinated + 500 non-hallucinated) with seed=42. The 20% test split is identical to variant testing → Wikipedia held-out AUROC is directly comparable.
* **HalluShift on MIND data.** Her original pipeline extracts features during *generation*. We adapt by extracting on the fixed MIND text (treating it as a generation trace) — features are averaged over all token positions in the text. Documented deviation.
* **EigenScore is unsupervised.** No training. The score is computed directly per query at evaluation time. Wikipedia held-out evaluation is done by re-generating from the text's first half.
* **Multi-task eval (Stage 8).** Same 7 datasets as `all_variants.ipynb`. For each baseline × each dataset × each sample: extract baseline-specific feature, classify with the trained head, record metrics.
* **eager attention.** Same `attn_implementation="eager"` fix from `all_variants.ipynb` so attention weights are real tensors (HaloScope needs them; HalluShift definitely needs them).

## Resume-safe stages

| # | Stage | Output (skipped on rerun if exists) |
|---|---|---|
| 1 | Setup + load dataset_full.json | (in memory) |
| 2 | Feature cache: layer-7 hidden, last-layer hidden, HalluShift 31-d, per record | `llama2_7b_baseline_feature_cache.json` |
| 3 | Train SAPLMA | `llama2_7b_saplma_best.pth` |
| 4 | Train HaloScope (with layer/k/sign/threshold sweep) | `llama2_7b_haloscope_best.pth` |
| 5 | Train HalluShift | `llama2_7b_hallushift_best.pth` |
| 6 | Wikipedia held-out eval for all 4 | (in memory) |
| 7 | Load 7 multi-task datasets | (in memory) |
| 8 | Multi-task eval per baseline | (in memory) |
| 9 | Consolidated dump | `llama2_7b_baselines_results.json` |

Paste `llama2_7b_baselines_results.json` back to the assistant for cross-method comparison against your variant results.


In [ ]:
# =============================================================================
# BLOCK 0: PIP INSTALLS
# =============================================================================
!pip install -q -U "transformers>=4.45" "tokenizers>=0.19" "accelerate>=0.30"
!pip install -q -U datasets nltk scikit-learn tqdm sentence-transformers scipy


In [ ]:
# =============================================================================
# BLOCK 1 (STAGE 1): SETUP + LOAD dataset_full.json
# =============================================================================
import os, sys, gc, json, random, math, time, datetime, platform, traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

MODEL_NAME       = "meta-llama/Llama-2-7b-hf"
MODEL_TAG        = "kaggle_llama2_7b"
SEED             = 42
DTYPE            = torch.float16 if torch.cuda.is_available() else torch.float32
TEST_FRAC        = 0.20
MAX_GEN_NEW      = 48

# Baseline-specific hyperparameters
SAPLMA_LAYER     = 16    # ~16/32 ≈ 0.50 of Llama-2-7B depth (matches paper sweet spot)
SAPLMA_EPOCHS    = 5
SAPLMA_LR        = 1e-3
SAPLMA_BATCH     = 32

HALOSCOPE_LAYERS_TO_TRY = list(range(1, 33))      # sweep all Llama-2-7B layers
HALOSCOPE_K_TO_TRY      = [1, 2, 3, 5, 8]         # top-k singular vectors
HALOSCOPE_THRESHOLDS    = [round(t, 3) for t in np.linspace(0.05, 0.95, 19)]
HALOSCOPE_PROBE_EPOCHS  = 50
HALOSCOPE_PROBE_LR      = 0.05
HALOSCOPE_PROBE_BATCH   = 512
HALOSCOPE_HIDDEN        = 1024
HALOSCOPE_WD            = 3e-4

EIGENSCORE_K_WIKI       = 10                       # paper default K
EIGENSCORE_K_DOWNSTREAM = 3                        # cost reduction for 7-dataset eval
EIGENSCORE_LAYER        = 16    # middle layer of Llama-2-7B (32//2) — INSIDE paper convention
EIGENSCORE_RIDGE        = 1e-3                     # alpha in the paper
EIGENSCORE_DS_CAP       = 100                      # subsample cap per downstream ds (EigenScore only)
EIGENSCORE_TOP_P        = 0.99
EIGENSCORE_TOP_K        = 10
EIGENSCORE_TEMP         = 0.5
EIGENSCORE_MAX_NEW      = 32

HALLUSHIFT_R            = 2                        # layer offset for Wasserstein/cosine
HALLUSHIFT_BATCH        = 16
HALLUSHIFT_LR           = 1e-4
HALLUSHIFT_WD           = 1e-4
HALLUSHIFT_EPOCHS       = 100
HALLUSHIFT_PATIENCE     = 10
HALLUSHIFT_LOSS_ALPHA   = 0.4

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PATH_DATASET_FULL   = f"{MODEL_TAG}_dataset_full.json"
PATH_FEATURE_CACHE  = f"{MODEL_TAG}_baseline_feature_cache.json"
PATH_RESULTS        = f"{MODEL_TAG}_baselines_results.json"

PATH_SAPLMA         = f"{MODEL_TAG}_saplma_best.pth"
PATH_HALOSCOPE      = f"{MODEL_TAG}_haloscope_best.pth"
PATH_HALLUSHIFT     = f"{MODEL_TAG}_hallushift_best.pth"

# Diagnostics (will become the output JSON)
DIAG = {
    "schema_version": "baselines-sota-1.0",
    "model_tag":  MODEL_TAG,
    "model_name": MODEL_NAME,
    "config": {
        "seed": SEED, "dtype": str(DTYPE), "test_frac": TEST_FRAC,
        "saplma_layer": SAPLMA_LAYER,
        "eigenscore_K_wiki": EIGENSCORE_K_WIKI,
        "eigenscore_K_downstream": EIGENSCORE_K_DOWNSTREAM,
        "eigenscore_ds_cap": EIGENSCORE_DS_CAP,
        "eigenscore_layer": EIGENSCORE_LAYER,
        "eigenscore_ridge": EIGENSCORE_RIDGE,
        "hallushift_r": HALLUSHIFT_R,
        "hallushift_lr": HALLUSHIFT_LR,
        "hallushift_loss_alpha": HALLUSHIFT_LOSS_ALPHA,
    },
    "env": {
        "python":   sys.version.split()[0],
        "platform": platform.platform(),
        "torch":    torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device":   (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
        "free_vram_gb":  (round(torch.cuda.mem_get_info()[0]/1e9, 2) if torch.cuda.is_available() else None),
    },
    "library_versions": {},
    "stage_timings_sec": {},
    "baselines": {},      # populated in Stage 6 + Stage 8
    "downstream_datasets": {},
    "errors": [],
}
for _mod in ["transformers", "datasets", "accelerate", "sklearn", "scipy", "numpy"]:
    try:
        m = __import__(_mod)
        DIAG["library_versions"][_mod] = getattr(m, "__version__", "n/a")
    except Exception as e:
        DIAG["library_versions"][_mod] = f"IMPORT_FAILED: {e}"

# Load dataset_full.json (produced by all_variants.ipynb's Stage 2)
if not os.path.exists(PATH_DATASET_FULL):
    raise SystemExit(
        f"[ERROR] {PATH_DATASET_FULL} not found. Run all_variants.ipynb first "
        f"(Stage 2 produces this file). Or upload it from your previous run.")
with open(PATH_DATASET_FULL, "r") as f:
    records_full = json.load(f)
print(f"✓ loaded {len(records_full)} records from {PATH_DATASET_FULL}")
print(f"  label dist: y=0: {sum(1 for r in records_full if r['label']==0)}   "
      f"y=1: {sum(1 for r in records_full if r['label']==1)}")

# 80/20 split — SAME seed as all_variants so AUROCs are directly comparable
random.seed(SEED)
records_shuf = list(records_full)
random.shuffle(records_shuf)
split_idx = int((1.0 - TEST_FRAC) * len(records_shuf))
train_recs = records_shuf[:split_idx]
test_recs  = records_shuf[split_idx:]
print(f"  train n = {len(train_recs)}   test n = {len(test_recs)}")
DIAG["n_train"] = len(train_recs); DIAG["n_test"] = len(test_recs)


In [ ]:
# =============================================================================
# BLOCK 2: LAZY LLM LOADER + ALWAYS-DEFINED FEATURE HELPERS
# =============================================================================
_MODEL = {"tokenizer": None, "model": None}

def need_llm():
    if _MODEL["model"] is not None:
        return _MODEL
    from transformers import AutoTokenizer, AutoModelForCausalLM
    print(f"Loading {MODEL_NAME} (dtype={DTYPE}) ...")
    _t0 = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    # eager attention — required for HalluShift (uses output_attentions=True)
    load_kwargs = dict(torch_dtype=DTYPE,
                       device_map="auto" if torch.cuda.is_available() else None,
                       attn_implementation="eager")
    try:
        mdl = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    except (TypeError, ValueError) as e:
        print(f"[warn] retry load without attn_implementation: {e}")
        load_kwargs.pop("attn_implementation", None)
        mdl = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    mdl.eval()
    for p in mdl.parameters(): p.requires_grad = False
    DIAG["stage_timings_sec"]["model_load"] = round(time.perf_counter() - _t0, 2)
    DIAG["model_info"] = {
        "hidden_dim":    int(mdl.config.hidden_size),
        "n_layers":      int(mdl.config.num_hidden_layers),
        "n_heads":       int(getattr(mdl.config, "num_attention_heads", -1)),
        "vocab_size":    int(mdl.config.vocab_size),
        "max_pos_embed": int(getattr(mdl.config, "max_position_embeddings", -1)),
    }
    _MODEL["tokenizer"] = tok; _MODEL["model"] = mdl
    print(f"✓ Llama-2-7B loaded in {DIAG['stage_timings_sec']['model_load']}s  hidden_dim={mdl.config.hidden_size}  layers={mdl.config.num_hidden_layers}")
    return _MODEL


# ---- HalluShift's normalized_entropy / count_low_probs / mean_gradient ----
def _normalized_entropy(seq):
    seq = np.asarray(seq, dtype=np.float64)
    if seq.size == 0: return 0.0
    seq = seq / (seq.sum() + 1e-12)
    H = -(seq * np.log(seq.clip(min=1e-12))).sum()
    return float(H / max(np.log(seq.size), 1e-12))

def _count_low_probs(seq, thr=0.1):
    seq = np.asarray(seq, dtype=np.float64)
    return float((seq < thr).mean() if seq.size else 0.0)

def _mean_gradient(seq):
    seq = np.asarray(seq, dtype=np.float64)
    if seq.size < 2: return 0.0
    return float(np.mean(np.abs(np.diff(seq))))


# ---- Single forward pass that extracts EVERY baseline-needed feature ----
@torch.no_grad()
def extract_all_baseline_features(text):
    """Returns:
        saplma_h7:    [hidden_dim] tensor — layer-7 last-token hidden state
        haloscope_per_layer: [n_layers+1, hidden_dim] tensor — last-token hidden, every layer
        hallushift_feats: [31] np.ndarray — 5+5+5+5 dist/sim + 11 token-prob
    """
    from scipy.stats import wasserstein_distance
    LL = need_llm()
    tokenizer, model = LL["tokenizer"], LL["model"]
    enc = tokenizer(text.strip(), return_tensors="pt", truncation=True,
                    max_length=getattr(model.config, "max_position_embeddings", 4096)).to(model.device)
    out = model(**enc, output_hidden_states=True, output_attentions=True, use_cache=False)
    hs  = out.hidden_states            # tuple length (L+1) — [embed, layer1, ..., layerL]
    attn = out.attentions              # tuple length L — [1, n_heads, T, T] per layer
    logits = out.logits                # [1, T, V]
    L = len(hs) - 1                    # number of transformer layers (model layer count)
    T = enc.input_ids.shape[1]

    # SAPLMA: layer SAPLMA_LAYER, last token, single 768-d vector
    saplma_h = hs[SAPLMA_LAYER][0, -1, :].float().cpu()

    # HaloScope: all layers, last token — cached so we can sweep layers
    haloscope_per_layer = torch.stack([hs[ell][0, -1, :].float().cpu() for ell in range(len(hs))], dim=0)

    # ---------------- HalluShift 31-d feature vector ----------------
    # Sampled layer pairs: r=2, even-indexed layers for hidden states, odd-indexed for attentions
    hs_layer_idx = list(range(2, L + 1, 2))   # e.g. [2,4,6,8,10,12] for Llama-2-7B
    # 5 consecutive pairs from this list
    hs_pairs = [(hs_layer_idx[i], hs_layer_idx[i+1]) for i in range(len(hs_layer_idx)-1)]
    attn_layer_idx = list(range(1, L, 2))     # e.g. [1,3,5,7,9,11] for Llama-2-7B
    attn_pairs = [(attn_layer_idx[i], attn_layer_idx[i+1]) for i in range(len(attn_layer_idx)-1)]

    # Per-token averaging: collect distances/sims over all token positions
    def _per_pair_avg_wasserstein_hs(pair):
        # Closed-form 1D Wasserstein for distributions on the SAME integer
        # support: W_1 = sum_k |F_P(k) - F_Q(k)|. Vectorised across T.
        # Mathematically equivalent to the per-token scipy call (just faster).
        a, b = pair
        ha = hs[a][0].float()                   # [T, d]
        hb = hs[b][0].float()                   # [T, d]
        pa = torch.softmax(ha, dim=-1).cpu().numpy()   # [T, d]
        pb = torch.softmax(hb, dim=-1).cpu().numpy()   # [T, d]
        cdf_a = np.cumsum(pa, axis=-1)
        cdf_b = np.cumsum(pb, axis=-1)
        per_token_W = np.sum(np.abs(cdf_a - cdf_b), axis=-1)   # [T]
        return float(np.mean(per_token_W)) if per_token_W.size else 0.0

    def _per_pair_avg_cos_hs(pair):
        a, b = pair
        ha = hs[a][0].float()   # [T, d]
        hb = hs[b][0].float()   # [T, d]
        cos = F.cosine_similarity(ha, hb, dim=-1)
        return float(cos.mean().item())

    def _per_pair_avg_wasserstein_attn(pair):
        # Same closed-form trick as _per_pair_avg_wasserstein_hs, but on the
        # head-averaged attention matrices. Attention rows already sum to ~1.
        a, b = pair
        Aa = attn[a-1][0].float().mean(dim=0).cpu().numpy()   # [T, T]
        Ab = attn[b-1][0].float().mean(dim=0).cpu().numpy()   # [T, T]
        cdf_a = np.cumsum(Aa, axis=-1)
        cdf_b = np.cumsum(Ab, axis=-1)
        per_token_W = np.sum(np.abs(cdf_a - cdf_b), axis=-1)   # [T]
        return float(np.mean(per_token_W)) if per_token_W.size else 0.0

    def _per_pair_avg_cos_attn(pair):
        a, b = pair
        Aa = attn[a-1][0].float().mean(dim=0)   # [T, T]
        Ab = attn[b-1][0].float().mean(dim=0)   # [T, T]
        cos = F.cosine_similarity(Aa, Ab, dim=-1)
        return float(cos.mean().item())

    w_hs   = [_per_pair_avg_wasserstein_hs(p)   for p in hs_pairs]
    c_hs   = [_per_pair_avg_cos_hs(p)            for p in hs_pairs]
    w_attn = [_per_pair_avg_wasserstein_attn(p) for p in attn_pairs]
    c_attn = [_per_pair_avg_cos_attn(p)          for p in attn_pairs]

    # Pad / truncate to exactly 5 each (Llama-2-7B usually gives exactly 5 — but be safe)
    def _pad5(x):
        if len(x) >= 5: return x[:5]
        return list(x) + [0.0] * (5 - len(x))

    w_hs = _pad5(w_hs); c_hs = _pad5(c_hs); w_attn = _pad5(w_attn); c_attn = _pad5(c_attn)

    # Token-prob features: 11 scalars from per-token softmax
    probs = torch.softmax(logits[0].float(), dim=-1)  # [T, V]
    P_max = probs.max(dim=-1).values.cpu().numpy()
    P_min = probs.min(dim=-1).values.cpu().numpy()
    tp_feats = [
        float(np.min(P_max)),
        float(np.max(P_max - P_min)),
        _normalized_entropy(P_max),
        _normalized_entropy(P_min),
        _count_low_probs(P_max, 0.1),
        _count_low_probs(P_min, 0.1),
        _mean_gradient(P_max),
        _mean_gradient(P_min),
        float(np.percentile(P_max, 25)),
        float(np.percentile(P_max, 50)),
        float(np.percentile(P_max, 75)),
    ]

    hallushift_feats = np.array(w_hs + c_hs + w_attn + c_attn + tp_feats, dtype=np.float32)
    assert hallushift_feats.shape == (31,), f"expected 31-d, got {hallushift_feats.shape}"
    return saplma_h, haloscope_per_layer, hallushift_feats


print("✓ feature helpers defined")


In [ ]:
# =============================================================================
# BLOCK 3 (STAGE 2): BUILD BASELINE FEATURE CACHE
# Run Llama-2-7B ONCE per text in dataset_full → cache SAPLMA + HaloScope + HalluShift
# features. Reused by Stages 3-5 for training, Stage 6 for Wikipedia eval.
# =============================================================================
if os.path.exists(PATH_FEATURE_CACHE):
    print(f"[SKIP STAGE 2] {PATH_FEATURE_CACHE} exists — loading.")
    _t0 = time.perf_counter()
    with open(PATH_FEATURE_CACHE, "r") as _f:
        cache = json.load(_f)
    DIAG["stage_timings_sec"]["cache_skipped_load"] = round(time.perf_counter() - _t0, 2)
    saplma_h_arr     = np.array(cache["saplma_h"], dtype=np.float32)
    haloscope_arr    = np.array(cache["haloscope_per_layer"], dtype=np.float32)
    hallushift_arr   = np.array(cache["hallushift_feats"], dtype=np.float32)
    labels_arr       = np.array(cache["labels"], dtype=np.int64)
    print(f"  cached: SAPLMA {saplma_h_arr.shape}  HaloScope {haloscope_arr.shape}  HalluShift {hallushift_arr.shape}")
else:
    print(f"Building baseline feature cache for {len(records_full)} records ...")
    need_llm()
    from tqdm.auto import tqdm
    saplma_h_list = []
    haloscope_list = []
    hallushift_list = []
    labels_list = []
    failures = 0
    _t0 = time.perf_counter()
    for i, r in enumerate(tqdm(records_full, desc="extract_baselines")):
        try:
            saplma_h, haloscope_per_layer, hallushift_feats = extract_all_baseline_features(r["text"])
            saplma_h_list.append(saplma_h.numpy().tolist())
            haloscope_list.append(haloscope_per_layer.numpy().tolist())
            hallushift_list.append(hallushift_feats.tolist())
            labels_list.append(int(r["label"]))
        except Exception as e:
            failures += 1
            if failures == 1:
                print(f"\n[!!] FIRST extract_baseline FAILURE at row {i}:")
                print(f"     {type(e).__name__}: {e}")
                traceback.print_exc()
                print()
            DIAG["errors"].append(f"extract_baselines row {i}: {type(e).__name__}: {e}")
            # pad with zeros to preserve indexing
            hidden_dim = DIAG["model_info"]["hidden_dim"]
            n_layers = DIAG["model_info"]["n_layers"]
            saplma_h_list.append([0.0] * hidden_dim)
            haloscope_list.append([[0.0] * hidden_dim for _ in range(n_layers + 1)])
            hallushift_list.append([0.0] * 31)
            labels_list.append(int(r["label"]))
    DIAG["stage_timings_sec"]["feature_cache"] = round(time.perf_counter() - _t0, 2)
    DIAG["n_feature_failures"] = failures
    cache = {
        "saplma_h": saplma_h_list,
        "haloscope_per_layer": haloscope_list,
        "hallushift_feats": hallushift_list,
        "labels": labels_list,
    }
    with open(PATH_FEATURE_CACHE, "w") as _f:
        json.dump(cache, _f)
    saplma_h_arr   = np.array(saplma_h_list, dtype=np.float32)
    haloscope_arr  = np.array(haloscope_list, dtype=np.float32)
    hallushift_arr = np.array(hallushift_list, dtype=np.float32)
    labels_arr     = np.array(labels_list, dtype=np.int64)
    print(f"\n✓ wrote {PATH_FEATURE_CACHE} ({os.path.getsize(PATH_FEATURE_CACHE)/1e6:.2f} MB)  failures: {failures}")
    print(f"  feature_cache: {DIAG['stage_timings_sec']['feature_cache']} s")

# Build the same shuffled train/test indices used in Stage 1
random.seed(SEED)
_all_idx = list(range(len(records_full)))
random.shuffle(_all_idx)
_split = int((1.0 - TEST_FRAC) * len(_all_idx))
train_idx = np.array(_all_idx[:_split])
test_idx  = np.array(_all_idx[_split:])
print(f"  train/test indices ready: {len(train_idx)}/{len(test_idx)}")


In [ ]:
# =============================================================================
# BLOCK 4 (STAGE 3): TRAIN SAPLMA (Azaria & Mitchell 2023)
# 4-layer MLP probe on layer-SAPLMA_LAYER last-token hidden state. BCE, 5 epochs.
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SAPLMAProbe(nn.Module):
    """768 → 256 → 128 → 64 → 1, ReLU, no dropout. BCE loss."""
    def __init__(self, input_dim=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128),       nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x): return self.net(x)


print(f"Training SAPLMA on layer-{SAPLMA_LAYER} last-token hidden state ...")
_t0 = time.perf_counter()
X_tr = torch.from_numpy(saplma_h_arr[train_idx]).float().to(device)
y_tr = torch.from_numpy(labels_arr[train_idx]).float().to(device)
X_te = torch.from_numpy(saplma_h_arr[test_idx]).float().to(device)
y_te = torch.from_numpy(labels_arr[test_idx]).float().to(device)

saplma = SAPLMAProbe(input_dim=X_tr.shape[1]).to(device)
opt = torch.optim.Adam(saplma.parameters(), lr=SAPLMA_LR)
loss_fn = nn.BCEWithLogitsLoss()

for ep in range(SAPLMA_EPOCHS):
    saplma.train()
    # mini-batch with shuffle
    perm = torch.randperm(X_tr.shape[0], device=device)
    epoch_loss = 0.0
    for i in range(0, X_tr.shape[0], SAPLMA_BATCH):
        idx = perm[i:i+SAPLMA_BATCH]
        opt.zero_grad()
        out = saplma(X_tr[idx]).squeeze(-1)
        l = loss_fn(out, y_tr[idx])
        l.backward(); opt.step()
        epoch_loss += float(l.item())
    saplma.eval()
    with torch.no_grad():
        out = saplma(X_te).squeeze(-1)
        pred = (torch.sigmoid(out) > 0.5).float()
        acc = float((pred == y_te).float().mean().item())
    print(f"  epoch {ep+1}/{SAPLMA_EPOCHS}  loss={epoch_loss/max(1, X_tr.shape[0]//SAPLMA_BATCH):.4f}  test_acc={acc:.4f}")

torch.save({"model_state": saplma.state_dict(), "input_dim": X_tr.shape[1]}, PATH_SAPLMA)
DIAG["stage_timings_sec"]["saplma_train"] = round(time.perf_counter() - _t0, 2)
print(f"✓ saved {PATH_SAPLMA}   train time: {DIAG['stage_timings_sec']['saplma_train']}s")


In [ ]:
# =============================================================================
# BLOCK 5 (STAGE 4): TRAIN HaloScope (Du et al. NeurIPS 2024)
# Steps:
#   1. For each candidate layer ell, compute centered last-token embeddings.
#   2. Truncated SVD → top-k singular vectors.
#   3. Spectral score per sample (weighted by sigma).
#   4. Threshold spectral score at percentile t → pseudo-labels.
#   5. Train non-linear probe (768 → 1024 → 1) on pseudo-labels via BCE + SGD.
#   6. Sweep (layer, k, sign, threshold) on a validation subset (last 100 of train).
#   7. Pick combo with highest val AUROC; save that probe.
# =============================================================================
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class HaloProbe(nn.Module):
    def __init__(self, d=768, h=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, h), nn.ReLU(),
            nn.Linear(h, 1),
        )
    def forward(self, x): return self.net(x)


def _haloscope_spectral_score(H, k, sign=+1):
    """H: [N, d] centered. Returns [N] spectral scores."""
    U, S, Vt = np.linalg.svd(H, full_matrices=False)   # Vt: [min(N,d), d]
    V_top = Vt[:k, :]            # [k, d]
    S_top = S[:k]                # [k]
    proj  = S_top[:, None] * V_top  # weighted-svd: [k, d]
    # score = sqrt( sum_j (mean_dim(H @ proj.T))^2 )
    # H @ proj.T = [N, k]
    M = H @ proj.T               # [N, k]
    M_mean = M.mean(axis=1, keepdims=True)  # [N, 1]
    scores = np.sqrt(np.sum(M_mean ** 2, axis=1))  # [N]
    return sign * scores


# Use last 100 of train as validation (to pick layer/k/sign/threshold)
val_size = min(100, len(train_idx) // 5)
tr_idx_inner = train_idx[:-val_size]
val_idx_inner = train_idx[-val_size:]

print(f"HaloScope sweep over layers × k × sign × threshold ...")
print(f"  inner train: {len(tr_idx_inner)}   val: {len(val_idx_inner)}   test: {len(test_idx)}")

best = {"val_auc": -1, "layer": None, "k": None, "sign": None, "thr": None,
         "probe_state": None, "input_dim": None}
_t0 = time.perf_counter()

# We sweep, but for cost reasons only train a probe at the chosen (layer, k, sign, thr).
# First pass: compute spectral val-AUC for each (layer, k, sign) without thresholding —
# pick best combo, then sweep threshold for the probe.
sweep_log = []
for ell in HALOSCOPE_LAYERS_TO_TRY:
    H_train = haloscope_arr[tr_idx_inner, ell, :]
    H_val   = haloscope_arr[val_idx_inner, ell, :]
    H_train_c = H_train - H_train.mean(axis=0, keepdims=True)
    H_val_c   = H_val - H_train.mean(axis=0, keepdims=True)
    for k in HALOSCOPE_K_TO_TRY:
        # Compute spectral scores
        try:
            sc_train = _haloscope_spectral_score(H_train_c, k, sign=+1)
            sc_val   = _haloscope_spectral_score(H_val_c, k, sign=+1)  # use train's projection
        except Exception:
            continue
        # Try both signs and pick whichever maximises val AUROC against TRUE labels
        for sign in [+1, -1]:
            y_val = labels_arr[val_idx_inner]
            sc_val_signed = sign * sc_val
            try:
                auc = roc_auc_score(y_val, sc_val_signed)
            except ValueError:
                continue
            sweep_log.append({"layer": ell, "k": k, "sign": sign, "spectral_val_auc": float(auc)})
            if auc > best["val_auc"]:
                best.update({"val_auc": float(auc), "layer": int(ell),
                              "k": int(k), "sign": int(sign), "thr": None,
                              "spectral_val_auc": float(auc)})

print(f"  best spectral-only: layer={best['layer']}  k={best['k']}  sign={best['sign']}  val_auc={best['val_auc']:.4f}")

# Now use the best (layer, k, sign) to compute pseudo-labels and train a probe.
ell = best["layer"]; k = best["k"]; sign = best["sign"]
H_full_train = haloscope_arr[tr_idx_inner, ell, :]
H_centered = H_full_train - H_full_train.mean(axis=0, keepdims=True)
sc_train = sign * _haloscope_spectral_score(H_centered, k)

best_thr_auc = -1
for thr in HALOSCOPE_THRESHOLDS:
    pseudo = (sc_train > np.quantile(sc_train, thr)).astype(np.float32)
    # Quick: probe-free CHECK — does pseudo split val well?
    # Train probe with this thr
    X = torch.from_numpy(H_full_train).float().to(device)
    y = torch.from_numpy(pseudo).float().to(device)
    probe = HaloProbe(d=X.shape[1], h=HALOSCOPE_HIDDEN).to(device)
    opt = torch.optim.SGD(probe.parameters(), lr=HALOSCOPE_PROBE_LR,
                           weight_decay=HALOSCOPE_WD, momentum=0.9)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HALOSCOPE_PROBE_EPOCHS)
    loss_fn = nn.BCEWithLogitsLoss()
    for ep in range(HALOSCOPE_PROBE_EPOCHS):
        probe.train()
        perm = torch.randperm(X.shape[0], device=device)
        for i in range(0, X.shape[0], HALOSCOPE_PROBE_BATCH):
            bi = perm[i:i+HALOSCOPE_PROBE_BATCH]
            opt.zero_grad()
            out = probe(X[bi]).squeeze(-1)
            l = loss_fn(out, y[bi])
            l.backward(); opt.step()
        sched.step()
    probe.eval()
    with torch.no_grad():
        H_val = haloscope_arr[val_idx_inner, ell, :]
        X_val = torch.from_numpy(H_val).float().to(device)
        prob_val = torch.sigmoid(probe(X_val).squeeze(-1)).cpu().numpy()
        y_val = labels_arr[val_idx_inner]
    try:
        val_auc = roc_auc_score(y_val, prob_val)
    except ValueError:
        val_auc = float("nan")
    if val_auc > best_thr_auc:
        best_thr_auc = val_auc
        best["thr"] = float(thr)
        best["probe_state"] = {k_: v.cpu().clone() for k_, v in probe.state_dict().items()}
        best["input_dim"] = int(X.shape[1])
        best["val_auc_probe"] = float(val_auc)

print(f"  best probe: thr={best['thr']}  val_auc_probe={best['val_auc_probe']:.4f}")
DIAG["stage_timings_sec"]["haloscope_train"] = round(time.perf_counter() - _t0, 2)

# Save
torch.save({
    "probe_state":    best["probe_state"],
    "best_layer":     best["layer"],
    "best_k":         best["k"],
    "best_sign":      best["sign"],
    "best_threshold": best["thr"],
    "input_dim":      best["input_dim"],
    "val_auc_probe":  best["val_auc_probe"],
    "spectral_val_auc": best["spectral_val_auc"],
    "centroid":       H_full_train.mean(axis=0).tolist(),
    "svd_projection": None,    # will recompute at eval — small cost
}, PATH_HALOSCOPE)
DIAG["baselines"]["HaloScope"] = {
    "best_layer": best["layer"], "best_k": best["k"],
    "best_sign": best["sign"], "best_threshold": best["thr"],
    "val_auc_spectral": best["spectral_val_auc"],
    "val_auc_probe":    best["val_auc_probe"],
}
print(f"✓ saved {PATH_HALOSCOPE}  train time: {DIAG['stage_timings_sec']['haloscope_train']}s")


In [ ]:
# =============================================================================
# BLOCK 6 (STAGE 5): TRAIN HalluShift (Dasgupta IJCNN 2025)
# CombinedNN: 4 parallel embedding heads → concat with 11 token-prob → 2-layer head
# Loss: BCEWithLogits + 0.4 * (1 - accuracy)
# AdamW lr=1e-4, ReduceLROnPlateau on val AUC, early stop patience=10
# =============================================================================
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class _FeatureEmbeddingNN(nn.Module):
    def __init__(self, d_in, d_out, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_in), nn.Dropout(dropout),
            nn.Linear(d_in, d_out),
        )
    def forward(self, x): return self.net(x)


class HalluShiftCombinedNN(nn.Module):
    """4 parallel embedding heads + 2-layer head w/ sigmoid output."""
    def __init__(self, n_dist_per_block=5, n_prob=11):
        super().__init__()
        d_emb = max(1, n_dist_per_block // 2)
        self.emb_w_hs   = _FeatureEmbeddingNN(n_dist_per_block, d_emb)
        self.emb_c_hs   = _FeatureEmbeddingNN(n_dist_per_block, d_emb)
        self.emb_w_attn = _FeatureEmbeddingNN(n_dist_per_block, d_emb)
        self.emb_c_attn = _FeatureEmbeddingNN(n_dist_per_block, d_emb)
        d_combined = 4 * d_emb + n_prob
        d_hidden   = max(1, 2 * d_emb)
        self.head = nn.Sequential(
            nn.LayerNorm(d_combined), nn.Dropout(0.2),
            nn.Linear(d_combined, d_hidden),
            nn.LayerNorm(d_hidden),    nn.Dropout(0.2),
            nn.Linear(d_hidden, 1),
        )
    def forward(self, x):
        # x: [B, 31] — first 5 are w_hs, then c_hs, w_attn, c_attn, then 11 token-prob
        e1 = self.emb_w_hs  (x[:, 0:5])
        e2 = self.emb_c_hs  (x[:, 5:10])
        e3 = self.emb_w_attn(x[:, 10:15])
        e4 = self.emb_c_attn(x[:, 15:20])
        z  = torch.cat([e1, e2, e3, e4, x[:, 20:31]], dim=1)
        return self.head(z)


def accuracy_improvement_loss(logits, targets, alpha=0.4):
    """BCEWithLogits + alpha * (1 - accuracy). Matches HalluShift's recipe."""
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    pred = (torch.sigmoid(logits) > 0.5).float()
    acc = (pred == targets).float().mean()
    return bce + alpha * (1.0 - acc)


print("Training HalluShift CombinedNN ...")
_t0 = time.perf_counter()

# Inner train/val (matches her train_test_split)
tr_idx_inner = train_idx[:-100]
val_idx_inner = train_idx[-100:]

scaler = StandardScaler().fit(hallushift_arr[tr_idx_inner])
X_tr = scaler.transform(hallushift_arr[tr_idx_inner]).astype(np.float32)
X_va = scaler.transform(hallushift_arr[val_idx_inner]).astype(np.float32)
X_te = scaler.transform(hallushift_arr[test_idx]).astype(np.float32)
y_tr = labels_arr[tr_idx_inner].astype(np.float32)
y_va = labels_arr[val_idx_inner].astype(np.float32)
y_te = labels_arr[test_idx].astype(np.float32)

# Class-balanced sample weights (inverse frequency)
class_counts = np.bincount(y_tr.astype(int))
N = len(y_tr)
w = N / np.maximum(class_counts, 1)
sample_w = np.where(y_tr == 1, w[1], w[0]).astype(np.float32)

X_tr_t = torch.from_numpy(X_tr).to(device)
y_tr_t = torch.from_numpy(y_tr).to(device)
sw_t   = torch.from_numpy(sample_w).to(device)
X_va_t = torch.from_numpy(X_va).to(device)
y_va_t = torch.from_numpy(y_va).to(device)
X_te_t = torch.from_numpy(X_te).to(device)
y_te_t = torch.from_numpy(y_te).to(device)

model_hs = HalluShiftCombinedNN().to(device)
opt = torch.optim.AdamW(model_hs.parameters(), lr=HALLUSHIFT_LR, weight_decay=HALLUSHIFT_WD)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)

best_val = -1
patience_cnt = 0
best_state = None
for ep in range(HALLUSHIFT_EPOCHS):
    model_hs.train()
    perm = torch.randperm(X_tr_t.shape[0], device=device)
    for i in range(0, X_tr_t.shape[0], HALLUSHIFT_BATCH):
        bi = perm[i:i+HALLUSHIFT_BATCH]
        opt.zero_grad()
        out = model_hs(X_tr_t[bi]).squeeze(-1)
        loss_per = accuracy_improvement_loss(out, y_tr_t[bi], alpha=HALLUSHIFT_LOSS_ALPHA)
        loss = (loss_per * sw_t[bi]).mean()
        loss.backward(); opt.step()
    model_hs.eval()
    with torch.no_grad():
        prob_va = torch.sigmoid(model_hs(X_va_t).squeeze(-1)).cpu().numpy()
    try:
        val_auc = roc_auc_score(y_va, prob_va)
    except ValueError:
        val_auc = float("nan")
    sched.step(val_auc if val_auc == val_auc else 0)
    if val_auc > best_val:
        best_val = val_auc
        best_state = {k: v.cpu().clone() for k, v in model_hs.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= HALLUSHIFT_PATIENCE:
            print(f"  early stop at epoch {ep+1}")
            break

model_hs.load_state_dict(best_state)
torch.save({"model_state": best_state,
             "scaler_mean": scaler.mean_, "scaler_scale": scaler.scale_,
             "best_val_auc": float(best_val)}, PATH_HALLUSHIFT)
DIAG["stage_timings_sec"]["hallushift_train"] = round(time.perf_counter() - _t0, 2)
DIAG["baselines"]["HalluShift"] = {"best_val_auc": float(best_val)}
print(f"✓ saved {PATH_HALLUSHIFT}  val_auc={best_val:.4f}  train time: {DIAG['stage_timings_sec']['hallushift_train']}s")


In [ ]:
# =============================================================================
# BLOCK 6.5 — INTENTIONALLY EMPTY
# Lookback Lens baseline was removed 2026-06-01 (final baseline set = 4:
# SAPLMA, HaloScope, HalluShift, EigenScore). Cell preserved as a no-op so
# the 13-cell layout stays consistent across all 7 baselines_sota notebooks.
# =============================================================================


In [ ]:
# =============================================================================
# BLOCK 7 (STAGE 6): WIKIPEDIA HELD-OUT EVAL FOR ALL 4 BASELINES
# Same 80/20 test split as all_variants.ipynb → AUROCs directly comparable.
# EigenScore: uses K=EIGENSCORE_K_WIKI samples per test record, generated from
# the test text's first half as the "query".
# =============================================================================
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              roc_auc_score, confusion_matrix, brier_score_loss)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _metrics(y, p, pr):
    y = np.asarray(y); p = np.asarray(p); pr = np.asarray(pr)
    if len(y) == 0: return {"n": 0}
    acc = float(accuracy_score(y, p))
    prec, rec, f1, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    try: auc = float(roc_auc_score(y, pr)) if len(set(y.tolist())) > 1 else float("nan")
    except ValueError: auc = float("nan")
    try: brier = float(brier_score_loss(y, pr))
    except ValueError: brier = float("nan")
    cm = confusion_matrix(y, p, labels=[0, 1])
    return {"n": int(len(y)), "accuracy": acc,
            "precision": float(prec), "recall": float(rec),
            "f1": float(f1), "auc_roc": auc, "brier": brier,
            "cm": {"tn": int(cm[0,0]), "fp": int(cm[0,1]),
                   "fn": int(cm[1,0]), "tp": int(cm[1,1])},
            "label_distribution": {"y=0": int((y==0).sum()), "y=1": int((y==1).sum())},
            "prob_min": round(float(pr.min()) if len(pr) else 0.0, 4),
            "prob_max": round(float(pr.max()) if len(pr) else 0.0, 4),
            "prob_mean": round(float(pr.mean()) if len(pr) else 0.0, 4)}


# ---- SAPLMA Wikipedia ----
saplma_ckpt = torch.load(PATH_SAPLMA, weights_only=False, map_location=device)
saplma_eval = SAPLMAProbe(input_dim=saplma_ckpt["input_dim"]).to(device)
saplma_eval.load_state_dict(saplma_ckpt["model_state"]); saplma_eval.eval()
with torch.no_grad():
    Xte = torch.from_numpy(saplma_h_arr[test_idx]).float().to(device)
    prob = torch.sigmoid(saplma_eval(Xte).squeeze(-1)).cpu().numpy()
    pred = (prob > 0.5).astype(int)
DIAG["baselines"].setdefault("SAPLMA", {})["wikipedia_eval"] = _metrics(labels_arr[test_idx], pred, prob)

# ---- HaloScope Wikipedia ----
ck = torch.load(PATH_HALOSCOPE, weights_only=False, map_location=device)
ell_b, k_b, sign_b = ck["best_layer"], ck["best_k"], ck["best_sign"]
centroid = np.asarray(ck["centroid"], dtype=np.float32)
probe = HaloProbe(d=ck["input_dim"], h=HALOSCOPE_HIDDEN).to(device)
probe.load_state_dict(ck["probe_state"]); probe.eval()
with torch.no_grad():
    H_te = haloscope_arr[test_idx, ell_b, :]
    X_te_torch = torch.from_numpy(H_te).float().to(device)
    prob = torch.sigmoid(probe(X_te_torch).squeeze(-1)).cpu().numpy()
    pred = (prob > 0.5).astype(int)
DIAG["baselines"].setdefault("HaloScope", {})["wikipedia_eval"] = _metrics(labels_arr[test_idx], pred, prob)

# ---- HalluShift Wikipedia ----
ck = torch.load(PATH_HALLUSHIFT, weights_only=False, map_location=device)
sc_mean  = np.asarray(ck["scaler_mean"], dtype=np.float64)
sc_scale = np.asarray(ck["scaler_scale"], dtype=np.float64)
def _hs_scale(x):
    return ((x - sc_mean) / np.where(sc_scale > 1e-12, sc_scale, 1.0)).astype(np.float32)
model_hs_eval = HalluShiftCombinedNN().to(device)
model_hs_eval.load_state_dict(ck["model_state"]); model_hs_eval.eval()
with torch.no_grad():
    X_te = _hs_scale(hallushift_arr[test_idx])
    X_te_t = torch.from_numpy(X_te).to(device)
    prob = torch.sigmoid(model_hs_eval(X_te_t).squeeze(-1)).cpu().numpy()
    pred = (prob > 0.5).astype(int)
DIAG["baselines"].setdefault("HalluShift", {})["wikipedia_eval"] = _metrics(labels_arr[test_idx], pred, prob)

# ---- EigenScore Wikipedia (unsupervised, K samples per query) ----
# IMPLEMENTATION NOTE: the official D2I-ai/eigenscore repo (generate.py) uses
# `num_return_sequences=num_generations_per_prompt` to produce K samples in
# ONE batched generate() call. We match that — it is both faithful to the
# repo AND ~5-6x faster than K sequential generate() calls on Llama-2-7B T4.
@torch.no_grad()
def _eigenscore_for_query(prompt, K, max_new):
    """K stochastic samples (batched) -> middle-layer last-token hidden -> K x K cov -> mean(log10(sigma))."""
    LL = need_llm()
    tokenizer, model_ = LL["tokenizer"], LL["model"]
    enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=getattr(model_.config, "max_position_embeddings", 4096) - max_new).to(model_.device)
    try:
        # ONE batched generate() call: num_return_sequences=K parallel samples
        out = model_.generate(
            **enc,
            max_new_tokens=max_new,
            do_sample=True, top_p=EIGENSCORE_TOP_P,
            top_k=EIGENSCORE_TOP_K, temperature=EIGENSCORE_TEMP,
            num_return_sequences=K,
            pad_token_id=tokenizer.eos_token_id,
            output_hidden_states=True, return_dict_in_generate=True,
        )
        # out.hidden_states is a tuple of length (n_generated_steps).
        # Each step's element is a tuple of (L+1) tensors of shape [K, current_T, d].
        # The LAST step's tensor at the chosen layer, at position -1, gives the
        # last-generated-token hidden state for each of the K samples.
        if hasattr(out, "hidden_states") and out.hidden_states:
            last_step_hs = out.hidden_states[-1]
            H_arr = last_step_hs[EIGENSCORE_LAYER][:, -1, :].float().cpu().numpy()   # [K, d]
        else:
            # Fallback: re-run a forward pass on each of K sequences (slow path)
            seqs = out.sequences   # [K, T_total]
            H_rows = []
            for k in range(K):
                full = model_(seqs[k:k+1], output_hidden_states=True, use_cache=False)
                H_rows.append(full.hidden_states[EIGENSCORE_LAYER][0, -1, :].float().cpu().numpy())
            H_arr = np.stack(H_rows, axis=0)
    except torch.cuda.OutOfMemoryError:
        # Memory-safe fallback for large K + long prompts: chunk K in halves
        torch.cuda.empty_cache()
        H_arr = None
        for chunk_K in [max(1, K // 2), max(1, K - K // 2)]:
            out = model_.generate(
                **enc,
                max_new_tokens=max_new,
                do_sample=True, top_p=EIGENSCORE_TOP_P,
                top_k=EIGENSCORE_TOP_K, temperature=EIGENSCORE_TEMP,
                num_return_sequences=chunk_K,
                pad_token_id=tokenizer.eos_token_id,
                output_hidden_states=True, return_dict_in_generate=True,
            )
            last_step_hs = out.hidden_states[-1]
            chunk = last_step_hs[EIGENSCORE_LAYER][:, -1, :].float().cpu().numpy()
            H_arr = chunk if H_arr is None else np.concatenate([H_arr, chunk], axis=0)
            torch.cuda.empty_cache()

    # K x K covariance (each sample row is a "variable" — matches the repo's torch.cov(E))
    if H_arr.shape[0] < 2:
        return float("nan")
    cov = np.cov(H_arr)                                       # [K, K]
    cov = cov + EIGENSCORE_RIDGE * np.eye(H_arr.shape[0])     # 1e-3 ridge (matches paper)
    try:
        s = np.linalg.svd(cov, compute_uv=False)
        score = float(np.mean(np.log10(s + 1e-12)))           # mean(log10(sigma)) — repo's `np.mean(np.log10(s))`
    except Exception:
        score = float("nan")
    return score


print(f"EigenScore Wikipedia eval (K={EIGENSCORE_K_WIKI}) on {len(test_idx)} samples ...")
_t0 = time.perf_counter()
es_scores = []
es_failures = 0
from tqdm.auto import tqdm
for i in tqdm(test_idx, desc="EigenScore wiki"):
    text = records_full[i]["text"]
    # Treat first sentence as the "query"; sample K continuations
    cut = text.find(". ")
    prompt = text[: cut + 2] if cut != -1 else text[: len(text)//2]
    try:
        s = _eigenscore_for_query(prompt, K=EIGENSCORE_K_WIKI, max_new=EIGENSCORE_MAX_NEW)
    except Exception:
        s = float("nan"); es_failures += 1
    es_scores.append(s)
es_scores = np.array(es_scores, dtype=np.float64)
# Higher score = more semantic diversity = more likely hallucinated
# Normalise to [0,1] via rank to use as pseudo-probability
finite = ~np.isnan(es_scores)
if finite.any():
    ranks = np.argsort(np.argsort(es_scores[finite])).astype(np.float64)
    pseudo_prob_finite = ranks / max(len(ranks) - 1, 1)
    pseudo_prob = np.zeros_like(es_scores)
    pseudo_prob[finite] = pseudo_prob_finite
    pseudo_prob[~finite] = 0.5
else:
    pseudo_prob = np.full_like(es_scores, 0.5)
pred = (pseudo_prob > 0.5).astype(int)
DIAG["baselines"].setdefault("EigenScore", {})["wikipedia_eval"] = _metrics(labels_arr[test_idx], pred, pseudo_prob)
DIAG["baselines"]["EigenScore"]["n_failures_wiki"] = int(es_failures)
DIAG["stage_timings_sec"]["eigenscore_wiki"] = round(time.perf_counter() - _t0, 2)

# Update final summary print to include 6 baselines
print("\nWikipedia held-out summary (4 baselines):")
print(f"{'Baseline':14s} {'AUROC':>8s} {'Acc':>7s} {'F1':>7s} {'Brier':>7s}")
print("-" * 60)
for name in ["SAPLMA", "HaloScope", "HalluShift", "EigenScore"]:
    if name in DIAG["baselines"] and "wikipedia_eval" in DIAG["baselines"][name]:
        m = DIAG["baselines"][name]["wikipedia_eval"]
        print(f"{name:14s} {m['auc_roc']:>8.4f} {m['accuracy']:>7.4f} {m['f1']:>7.4f} {m['brier']:>7.4f}")


In [ ]:
# =============================================================================
# BLOCK 8 (STAGE 7): LOAD 7 DOWNSTREAM DATASETS + MiniLM SCORER
# Same loaders + namespace fix as all_variants.ipynb. Same SUBSAMPLES.
# =============================================================================
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# Use the same 500/class scaling as the all_variants run
_TARGET_SAMPLES_EQ = 500
_scale = max(0.05, _TARGET_SAMPLES_EQ / 1000.0)
def _sub(n): return max(20, int(round(n * _scale)))
SUBSAMPLES = {"truthfulqa": None if _scale >= 0.5 else _sub(200),
              "triviaqa": _sub(500), "coqa": _sub(500), "tydiqa": _sub(500),
              "halueval_qa": _sub(1000), "halueval_summ": _sub(500),
              "halueval_dialog": _sub(1000),
              "nq_open": _sub(500), "hotpotqa": _sub(500), "popqa": _sub(1000)}
print(f"SUBSAMPLES (same as all_variants): {SUBSAMPLES}")

# Local-parquet shortcut: pick up files produced by 00_download_eval_datasets.ipynb
_LOCAL_PARQUET_CANDIDATES = [
    "eval_{label}.parquet",
    "./eval_datasets/eval_{label}.parquet",
    "/kaggle/input/dissertation-eval-datasets/eval_{label}.parquet",
    "/kaggle/input/eval-datasets/eval_{label}.parquet",
    "/kaggle/working/eval_{label}.parquet",
    "/content/drive/MyDrive/eval_datasets/eval_{label}.parquet",
    "/content/eval_{label}.parquet",
]

def _find_local_parquet(label):
    for tpl in _LOCAL_PARQUET_CANDIDATES:
        p = tpl.format(label=label)
        if os.path.exists(p):
            return p
    return None


def safe_load_first(*tries, subsample=None, label=""):
    # 1. Try local parquet first (fastest, no HF call)
    local = _find_local_parquet(label)
    if local:
        try:
            from datasets import Dataset
            ds = Dataset.from_parquet(local)
            if subsample and len(ds) > subsample:
                ds = ds.shuffle(seed=SEED).select(range(subsample))
            print(f"  ✓ {label}: {len(ds)} samples (LOCAL: {local})")
            DIAG["downstream_datasets"][label] = {
                "loaded": True, "n": int(len(ds)),
                "loader_idx_used": -1, "source": "local_parquet",
                "local_path": local,
            }
            return ds
        except Exception as e:
            print(f"  [warn] local {local} failed: {e} — falling back to HF download")
    # 2. Fall back to HuggingFace download
    last_err = None
    for i, fn in enumerate(tries):
        try:
            ds = fn()
            if subsample and len(ds) > subsample:
                ds = ds.shuffle(seed=SEED).select(range(subsample))
            print(f"  ✓ {label}: {len(ds)} samples (HF loader #{i})")
            DIAG["downstream_datasets"][label] = {"loaded": True, "n": int(len(ds)), "loader_idx_used": i, "source": "huggingface"}
            return ds
        except Exception as e:
            last_err = e; continue
    print(f"  ✗ {label}: FAILED — {last_err}")
    DIAG["downstream_datasets"][label] = {"loaded": False, "error": str(last_err)}
    DIAG["errors"].append(f"dataset_load[{label}]: {last_err}")
    return None

DATASETS = {}
DATASETS["truthfulqa"] = safe_load_first(
    lambda: load_dataset("truthfulqa/truthful_qa", "generation", split="validation"),
    lambda: load_dataset("truthful_qa", "generation", split="validation"),
    subsample=SUBSAMPLES["truthfulqa"], label="truthfulqa")
DATASETS["triviaqa"] = safe_load_first(
    lambda: load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation"),
    lambda: load_dataset("trivia_qa", "rc.nocontext", split="validation"),
    subsample=SUBSAMPLES["triviaqa"], label="triviaqa")
DATASETS["coqa"] = safe_load_first(
    lambda: load_dataset("stanfordnlp/coqa", split="validation"),
    subsample=SUBSAMPLES["coqa"], label="coqa")
def _load_tydi():
    ds = load_dataset("google-research-datasets/tydiqa", "secondary_task", split="validation")
    return ds.filter(lambda x: "english" in x.get("id", "").lower())
def _load_tydi_legacy():
    ds = load_dataset("tydiqa", "secondary_task", split="validation")
    return ds.filter(lambda x: "english" in x.get("id", "").lower())
DATASETS["tydiqa"] = safe_load_first(_load_tydi, _load_tydi_legacy,
    subsample=SUBSAMPLES["tydiqa"], label="tydiqa")
def _load_he(cfg):
    for cand in ("data", "train", "validation"):
        try: return load_dataset("pminervini/HaluEval", cfg, split=cand)
        except (ValueError, KeyError, FileNotFoundError): continue
    dd = load_dataset("pminervini/HaluEval", cfg); return dd[list(dd.keys())[0]]
for cfg, label in [("qa","halueval_qa"), ("summarization","halueval_summ"),
                   ("dialogue","halueval_dialog")]:
    DATASETS[label] = safe_load_first(
        lambda c=cfg: _load_he(c), subsample=SUBSAMPLES[label], label=label)

DATASETS["nq_open"] = safe_load_first(
    lambda: load_dataset("google-research-datasets/nq_open", split="validation"),
    lambda: load_dataset("nq_open", split="validation"),
    subsample=SUBSAMPLES["nq_open"], label="nq_open")
DATASETS["hotpotqa"] = safe_load_first(
    lambda: load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation", trust_remote_code=True),
    lambda: load_dataset("hotpot_qa", "distractor", split="validation", trust_remote_code=True),
    subsample=SUBSAMPLES["hotpotqa"], label="hotpotqa")
DATASETS["popqa"] = safe_load_first(
    lambda: load_dataset("akariasai/PopQA", split="test"),
    subsample=SUBSAMPLES["popqa"], label="popqa")

scorer = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                              device=str(device))
def score_match(gen, golds, threshold=0.5):
    if not golds: return 0
    if isinstance(golds, str): golds = [golds]
    embs = scorer.encode([gen] + list(golds), convert_to_numpy=True,
                          normalize_embeddings=True)
    sims = embs[0] @ embs[1:].T
    return 0 if float(sims.max()) >= threshold else 1
print("✓ MiniLM scorer ready")


In [ ]:
# =============================================================================
# BLOCK 9 (STAGE 8): MULTI-TASK EVAL — ALL 6 BASELINES × ALL 7 DATASETS
# Efficiency: per sample, ONE feature extraction → 3 supervised baselines share it.
# EigenScore needs K extra generations per sample → separate, capped subsamples.
# =============================================================================
from tqdm.auto import tqdm

LL = need_llm()
tokenizer, model = LL["tokenizer"], LL["model"]

@torch.no_grad()
def generate_short_answer(prompt, max_new=MAX_GEN_NEW):
    _max_pos = getattr(model.config, "max_position_embeddings", 4096)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=max(_max_pos - max_new, 256)).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new,
                          do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc.input_ids.shape[1]:],
                            skip_special_tokens=True).strip()


def classify_supervised_three(text):
    """Run Llama-2-7B once on `text`, extract baseline features, return per-baseline (pred, prob)."""
    saplma_h, haloscope_per_layer, hallushift_feats = extract_all_baseline_features(text)
    out = {}
    # SAPLMA
    Xt = saplma_h.to(device).unsqueeze(0)
    with torch.no_grad():
        prob = float(torch.sigmoid(saplma_eval(Xt).squeeze(-1))[0].item())
    out["SAPLMA"] = (int(prob > 0.5), prob)
    # HaloScope
    ck = torch.load(PATH_HALOSCOPE, weights_only=False, map_location=device) if "haloscope_ck" not in globals() else haloscope_ck
    globals()["haloscope_ck"] = ck
    H_te = haloscope_per_layer[ck["best_layer"]].numpy()[None, :]
    Xt = torch.from_numpy(H_te).float().to(device)
    with torch.no_grad():
        prob = float(torch.sigmoid(probe(Xt).squeeze(-1))[0].item())
    out["HaloScope"] = (int(prob > 0.5), prob)
    # HalluShift
    X = _hs_scale(hallushift_feats[None, :])
    Xt = torch.from_numpy(X).to(device)
    with torch.no_grad():
        prob = float(torch.sigmoid(model_hs_eval(Xt).squeeze(-1))[0].item())
    out["HalluShift"] = (int(prob > 0.5), prob)
    return out


def _accumulate(per_baseline_acc, baseline_preds, gold_label):
    for name, (pp, prob) in baseline_preds.items():
        per_baseline_acc[name]["y"].append(gold_label)
        per_baseline_acc[name]["p"].append(pp)
        per_baseline_acc[name]["pr"].append(prob)


def eval_supervised_open(ds, prompt_fn, gold_fn, dsname):
    per = {b: {"y": [], "p": [], "pr": []} for b in ["SAPLMA", "HaloScope", "HalluShift"]}
    fails = 0
    for s in tqdm(ds, desc=f"{dsname} (sup)"):
        try:
            prompt = prompt_fn(s)
            gen = generate_short_answer(prompt)
            gold = gold_fn(s)
            label = score_match(gen, gold)
            preds = classify_supervised_three((prompt + " " + gen).strip())
            _accumulate(per, preds, label)
        except Exception:
            fails += 1; continue
    return per, fails


def eval_supervised_he(ds, prompt_fn, right_key, wrong_key, dsname):
    per = {b: {"y": [], "p": [], "pr": []} for b in ["SAPLMA", "HaloScope", "HalluShift"]}
    fails = 0
    for s in tqdm(ds, desc=f"{dsname} (sup)"):
        try:
            prompt = prompt_fn(s)
            for ak, gl in [(right_key, 0), (wrong_key, 1)]:
                ans = s[ak]
                preds = classify_supervised_three((prompt + " " + ans).strip())
                _accumulate(per, preds, gl)
        except Exception:
            fails += 1; continue
    return per, fails


def eval_eigenscore_open(ds, prompt_fn, gold_fn, dsname, K, cap):
    """Open-QA EigenScore: subsample to cap, K samples per query."""
    selected = ds.select(range(min(cap, len(ds))))
    scores, labels = [], []
    fails = 0
    for s in tqdm(selected, desc=f"{dsname} (ES)"):
        try:
            prompt = prompt_fn(s)
            # Use greedy as the "answer" against gold to derive a label
            gen = generate_short_answer(prompt)
            gold = gold_fn(s)
            label = score_match(gen, gold)
            es = _eigenscore_for_query(prompt, K=K, max_new=EIGENSCORE_MAX_NEW)
            scores.append(es); labels.append(label)
        except Exception:
            fails += 1; continue
    return np.array(scores, dtype=np.float64), np.array(labels, dtype=np.int64), fails


def eval_eigenscore_he(ds, prompt_fn, right_key, wrong_key, dsname, K, cap):
    """HaluEval EigenScore: K samples from PROMPT only; label by which answer is closer."""
    selected = ds.select(range(min(cap, len(ds))))
    scores, labels = [], []
    fails = 0
    for s in tqdm(selected, desc=f"{dsname} (ES)"):
        try:
            prompt = prompt_fn(s)
            es = _eigenscore_for_query(prompt, K=K, max_new=EIGENSCORE_MAX_NEW)
            # 2 rows per sample: same ES score, but gold label differs
            scores.append(es); labels.append(0)
            scores.append(es); labels.append(1)
        except Exception:
            fails += 1; continue
    return np.array(scores, dtype=np.float64), np.array(labels, dtype=np.int64), fails


def _eigenscore_to_metrics(scores, labels):
    finite = ~np.isnan(scores)
    if finite.sum() < 2:
        return {"n": int(len(labels))}
    ranks = np.argsort(np.argsort(scores[finite])).astype(np.float64)
    pseudo = np.zeros_like(scores)
    pseudo[finite] = ranks / max(len(ranks) - 1, 1)
    pseudo[~finite] = 0.5
    pred = (pseudo > 0.5).astype(int)
    return _metrics(labels, pred, pseudo)


# Build EVAL_TASKS
EVAL_TASKS = []
if DATASETS["truthfulqa"] is not None:
    EVAL_TASKS.append(("truthfulqa", "open", DATASETS["truthfulqa"],
        lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        lambda s: s.get("correct_answers", []),
        None, None))
if DATASETS["triviaqa"] is not None:
    EVAL_TASKS.append(("triviaqa", "open", DATASETS["triviaqa"],
        lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        lambda s: list(s["answer"].get("aliases", [])) + [s["answer"].get("value", "")],
        None, None))
if DATASETS["coqa"] is not None:
    def _cpr(s):
        ctx = s["story"][:600]
        q = s["questions"][0] if s["questions"] else ""
        return f"Context: {ctx} Q: {q} A:"
    def _cg(s):
        a = s["answers"]; return a["input_text"][0] if a["input_text"] else ""
    EVAL_TASKS.append(("coqa", "open", DATASETS["coqa"], _cpr, _cg, None, None))
if DATASETS["tydiqa"] is not None:
    EVAL_TASKS.append(("tydiqa", "open", DATASETS["tydiqa"],
        lambda s: f"Context: {s['context'][:600]} Q: {s['question']} A:",
        lambda s: s.get("answers", {}).get("text", []),
        None, None))
if DATASETS["halueval_qa"] is not None:
    EVAL_TASKS.append(("halueval_qa", "he", DATASETS["halueval_qa"],
        lambda s: f"Context: {s['knowledge'][:400]} Q: {s['question']} A:",
        None, "right_answer", "hallucinated_answer"))
if DATASETS["halueval_summ"] is not None:
    EVAL_TASKS.append(("halueval_summ", "he", DATASETS["halueval_summ"],
        lambda s: f"{s['document'][:500]} Summary:",
        None, "right_summary", "hallucinated_summary"))
if DATASETS["halueval_dialog"] is not None:
    EVAL_TASKS.append(("halueval_dialog", "he", DATASETS["halueval_dialog"],
        lambda s: f"Knowledge: {s['knowledge'][:300]}\nDialogue: {s['dialogue_history'][:300]}\n[Assistant]:",
        None, "right_response", "hallucinated_response"))

# ---- 3 ADDITIONAL DATASETS (2026-05-30): NQ-Open, HotpotQA, PopQA ----
if DATASETS.get("nq_open") is not None:
    EVAL_TASKS.append(("nq_open", "open", DATASETS["nq_open"],
        lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        lambda s: s.get("answer", []) if isinstance(s.get("answer"), list) else [s.get("answer", "")],
        None, None))
if DATASETS.get("hotpotqa") is not None:
    def _hotpot_pr_b(s):
        try:
            sents = s["context"]["sentences"]
            ctx = " ".join([" ".join(p) for p in sents[:3]])[:600]
        except Exception:
            ctx = ""
        return f"Context: {ctx} Q: {s['question']} A:"
    EVAL_TASKS.append(("hotpotqa", "open", DATASETS["hotpotqa"], _hotpot_pr_b,
        lambda s: [s.get("answer", "")], None, None))
if DATASETS.get("popqa") is not None:
    import json as _json_popqa
    def _popqa_gold_b(s):
        pa = s.get("possible_answers", "[]")
        if isinstance(pa, str):
            try: pa = _json_popqa.loads(pa)
            except Exception: pa = [pa]
        return pa if isinstance(pa, list) else [pa]
    EVAL_TASKS.append(("popqa", "open", DATASETS["popqa"],
        lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        _popqa_gold_b, None, None))


print(f"\nMulti-task eval — supervised 3 baselines + EigenScore on {len(EVAL_TASKS)} datasets ...")
_t0 = time.perf_counter()
multitask = {b: {} for b in ["SAPLMA", "HaloScope", "HalluShift", "EigenScore"]}
fails_per_ds = {}

for task in EVAL_TASKS:
    ds_name, ds_kind, ds, prompt_fn, gold_fn, rk, wk = task
    print(f"\n--- {ds_name} ({ds_kind}, n={len(ds)}) ---")
    if ds_kind == "open":
        per, fails = eval_supervised_open(ds, prompt_fn, gold_fn, ds_name)
        # EigenScore subsample
        es_scores, es_labels, es_fails = eval_eigenscore_open(
            ds, prompt_fn, gold_fn, ds_name, K=EIGENSCORE_K_DOWNSTREAM, cap=EIGENSCORE_DS_CAP)
    else:
        per, fails = eval_supervised_he(ds, prompt_fn, rk, wk, ds_name)
        es_scores, es_labels, es_fails = eval_eigenscore_he(
            ds, prompt_fn, rk, wk, ds_name, K=EIGENSCORE_K_DOWNSTREAM, cap=EIGENSCORE_DS_CAP)

    for b in ["SAPLMA", "HaloScope", "HalluShift"]:
        d = per[b]
        multitask[b][ds_name] = _metrics(d["y"], d["p"], d["pr"])
    multitask["EigenScore"][ds_name] = _eigenscore_to_metrics(es_scores, es_labels)
    fails_per_ds[ds_name] = {"sup": fails, "es": int(es_fails)}
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    # Print row
    print(f"  {'baseline':12s} {'AUROC':>7s} {'Acc':>7s} {'F1':>7s}")
    for b in ["SAPLMA", "HaloScope", "HalluShift", "EigenScore"]:
        m = multitask[b][ds_name]
        if m.get("n", 0) == 0:
            print(f"  {b:12s}     —       —       —")
        else:
            print(f"  {b:12s} {m.get('auc_roc', float('nan')):>7.3f} "
                  f"{m.get('accuracy', float('nan')):>7.3f} "
                  f"{m.get('f1', float('nan')):>7.3f}")

DIAG["stage_timings_sec"]["multitask_eval"] = round(time.perf_counter() - _t0, 2)
for b in ["SAPLMA", "HaloScope", "HalluShift", "EigenScore"]:
    DIAG["baselines"].setdefault(b, {})["multitask"] = multitask[b]
DIAG["downstream_datasets"]["n_eval_failures_per_dataset"] = fails_per_ds


In [ ]:
# =============================================================================
# BLOCK 10 (STAGE 9): CONSOLIDATED DUMP
# =============================================================================
DIAG["timestamp_utc"] = (datetime.datetime.now(datetime.timezone.utc)
                          .replace(microsecond=0).isoformat().replace("+00:00", "Z"))
DIAG["host"] = ("kaggle" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ
                else ("colab" if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
                      else "local"))
DIAG["stage_timings_sec"]["total"] = round(sum(DIAG["stage_timings_sec"].values()), 2)

with open(PATH_RESULTS, "w") as f:
    json.dump(DIAG, f, indent=2, default=str)
print(f"\n✓ wrote {PATH_RESULTS}  ({os.path.getsize(PATH_RESULTS)/1024:.1f} KB)")

# Summary 1 — Wikipedia
print("\n" + "=" * 70)
print(f"WIKIPEDIA HELD-OUT — {MODEL_TAG}")
print("=" * 70)
print(f"{'Baseline':12s} {'AUROC':>8s} {'Acc':>7s} {'F1':>7s} {'Brier':>7s}")
print("-" * 70)
for name in ["SAPLMA", "HaloScope", "HalluShift", "EigenScore"]:
    if name not in DIAG["baselines"] or "wikipedia_eval" not in DIAG["baselines"][name]:
        continue
    m = DIAG["baselines"][name]["wikipedia_eval"]
    print(f"{name:12s} {m['auc_roc']:>8.4f} {m['accuracy']:>7.4f} {m['f1']:>7.4f} {m['brier']:>7.4f}")
print("=" * 70)

# Summary 2 — multi-task AUROC matrix (rows=baselines, cols=datasets)
ds_order = ["truthfulqa","triviaqa","coqa","tydiqa","halueval_qa","halueval_summ","halueval_dialog","nq_open","hotpotqa","popqa"]
ds_short = {"truthfulqa":"TruthQA","triviaqa":"TrivQA","coqa":"CoQA","tydiqa":"TydiQA",
             "halueval_qa":"HE-QA","halueval_summ":"HE-Sum","halueval_dialog":"HE-Dial",
             "nq_open":"NQ","hotpotqa":"HotpotQA","popqa":"PopQA"}
print("\n" + "=" * 90)
print(f"MULTI-TASK AUROC — {MODEL_TAG}")
print("=" * 90)
hdr = f"{'Baseline':12s} "
for ds in ds_order: hdr += f"{ds_short[ds]:>9s} "
print(hdr + f"{'avg':>9s}")
print("-" * 90)
for name in ["SAPLMA", "HaloScope", "HalluShift", "EigenScore"]:
    if name not in DIAG["baselines"]:
        continue
    mt = DIAG["baselines"][name].get("multitask", {})
    row = f"{name:12s} "
    aurocs = []
    for ds in ds_order:
        m = mt.get(ds, {}); auc = m.get('auc_roc')
        if auc is not None and auc == auc:
            aurocs.append(auc)
            row += f"{auc:>9.3f} "
        else:
            row += f"{'—':>9s} "
    avg = sum(aurocs)/len(aurocs) if aurocs else float('nan')
    row += f"{avg:>9.3f}"
    print(row)
print("=" * 90)
print(f"\nTimings:")
for k, v in DIAG["stage_timings_sec"].items():
    print(f"  {k:30s} {v:>8.2f} s")

print(f"\nPaste {PATH_RESULTS} back to the assistant for comparison vs variants.")


# =============================================================================
# AUTO-DOWNLOAD outputs to user's local machine (Colab) or persist (Kaggle)
# =============================================================================
def _auto_download(file_patterns):
    import os, glob
    resolved = []
    for fp in file_patterns:
        if "*" in fp or "?" in fp:
            resolved.extend(sorted(glob.glob(fp)))
        elif os.path.exists(fp):
            resolved.append(fp)
    print("\nFINAL OUTPUTS")
    print("-" * 60)
    for f in resolved:
        print(f"  {f}   ({os.path.getsize(f)/1e6:.2f} MB)")
    if not resolved:
        return
    try:
        from google.colab import files as _colab_files
        print("\nColab detected -> triggering browser downloads ...")
        for f in resolved:
            try:    _colab_files.download(f)
            except Exception as e: print(f"  download failed for {f}: {e}")
        return
    except ImportError:
        pass
    if os.path.exists("/kaggle/working"):
        print("\nKaggle detected -> files persist in /kaggle/working/. Download via 'Output' panel.")
        return
    print(f"\nLocal Jupyter -> files are in {os.getcwd()}")


_auto_download([
    PATH_RESULTS,
    PATH_SAPLMA,
    PATH_HALOSCOPE,
    PATH_HALLUSHIFT,
])
